In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip install -q nltk rouge-score pandas

Mounted at /content/drive
  Preparing metadata (setup.py) ... done


In [ ]:
import json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from nltk.translate.bleu_score import (
    sentence_bleu,
    SmoothingFunction,
)
from rouge_score import rouge_scorer




RESULT_DIR = Path("/content/drive/MyDrive/cbt_results")

# This file is read only and will not be modified.
RESPONSES_PATH = RESULT_DIR / "responses_for_judge.json"

assert RESULT_DIR.exists(), (
    f"Result directory does not exist: {RESULT_DIR}"
)

assert RESPONSES_PATH.exists(), (
    f"Response file does not exist: {RESPONSES_PATH}"
)

print("Input file:", RESPONSES_PATH)

Input file: /content/drive/MyDrive/cbt_results/responses_for_judge.json


In [ ]:


with open(RESPONSES_PATH, "r", encoding="utf-8") as f:
    test_cases = json.load(f)

print(f"Loaded {len(test_cases)} single-turn cases")



MODEL_FIELDS = {
    "base": "base_response",
    "ft": "ft_response",
    "rag": "rag_response",
    "ft_rag": "ft_rag_response",
}




assert isinstance(test_cases, list), (
    "The response file must contain a JSON list."
)

assert len(test_cases) == 100, (
    f"Expected 100 test cases, but found {len(test_cases)}."
)

case_ids = [case.get("id") for case in test_cases]

assert len(case_ids) == len(set(case_ids)), (
    "Duplicate case IDs were detected."
)

category_counts = Counter(
    case.get("category") for case in test_cases
)

assert len(category_counts) == 20, (
    f"Expected 20 categories, but found {len(category_counts)}."
)

assert all(count == 5 for count in category_counts.values()), (
    f"Each category should contain five cases: {category_counts}"
)


missing_values = []

for case in test_cases:
    case_id = case.get("id")

    reference = case.get("reference")

    if not isinstance(reference, str) or not reference.strip():
        missing_values.append(
            {
                "id": case_id,
                "missing_field": "reference",
            }
        )

    for model_name, response_field in MODEL_FIELDS.items():
        response = case.get(response_field)

        if not isinstance(response, str) or not response.strip():
            missing_values.append(
                {
                    "id": case_id,
                    "model": model_name,
                    "missing_field": response_field,
                }
            )

if missing_values:
    print("Missing values detected:")
    display(pd.DataFrame(missing_values))

    raise ValueError(
        "Some test cases do not contain all four model responses."
    )


print("\nValidation passed.")
print("Number of cases:", len(test_cases))
print("Number of categories:", len(category_counts))
print("Responses per model:", len(test_cases))

print("\nCategory distribution:")
display(
    pd.DataFrame(
        sorted(category_counts.items()),
        columns=["Category", "Number of cases"],
    )
)

Loaded 100 single-turn cases

Validation passed.
Number of cases: 100
Number of categories: 20
Responses per model: 100

Category distribution:


,Category,Number of cases
0,anxiety,5
1,avoidance,5
2,burnout,5
3,catastrophizing,5
4,comparison,5
5,control,5
6,emotional_reasoning,5
7,future_worry,5
8,guilt,5
9,hopelessness,5


In [ ]:


# This matches the original Base/FT/FT+RAG implementation.
bleu_smoother = SmoothingFunction().method4

rouge_evaluator = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True,
)


def tokenize_for_bleu(text):
    """
    Apply the same preprocessing to references and model responses.
    """
    return text.lower().split()


def compute_lexical_metrics(reference, hypothesis):
    """
    Calculate sentence-level BLEU and ROUGE F-measures.
    """
    reference_tokens = tokenize_for_bleu(reference)
    hypothesis_tokens = tokenize_for_bleu(hypothesis)

    bleu_score = sentence_bleu(
        [reference_tokens],
        hypothesis_tokens,
        smoothing_function=bleu_smoother,
    )

    rouge_scores = rouge_evaluator.score(
        reference,
        hypothesis,
    )

    return {
        "bleu": float(bleu_score),
        "rouge1": float(
            rouge_scores["rouge1"].fmeasure
        ),
        "rouge2": float(
            rouge_scores["rouge2"].fmeasure
        ),
        "rougeL": float(
            rouge_scores["rougeL"].fmeasure
        ),
    }




results = []

for case in test_cases:
    record = {
        "id": case["id"],
        "category": case["category"],
    }

    reference = case["reference"]

    for model_name, response_field in MODEL_FIELDS.items():
        hypothesis = case[response_field]

        scores = compute_lexical_metrics(
            reference=reference,
            hypothesis=hypothesis,
        )

        for metric_name, metric_value in scores.items():
            record[
                f"{model_name}_{metric_name}"
            ] = metric_value

    results.append(record)


EXPECTED_METRIC_FIELDS = [
    f"{model_name}_{metric_name}"
    for model_name in MODEL_FIELDS
    for metric_name in [
        "bleu",
        "rouge1",
        "rouge2",
        "rougeL",
    ]
]

assert len(results) == 100

for record in results:
    for metric_field in EXPECTED_METRIC_FIELDS:
        assert metric_field in record, (
            f"Missing {metric_field} for case {record['id']}."
        )

        value = record[metric_field]

        assert np.isfinite(value), (
            f"Invalid value for {metric_field}, case {record['id']}."
        )

        assert 0.0 <= value <= 1.0, (
            f"Out-of-range value for {metric_field}, "
            f"case {record['id']}: {value}"
        )


print("All four model configurations were evaluated successfully.")
print("Number of per-case records:", len(results))
print("Metrics per model: BLEU, ROUGE-1, ROUGE-2, ROUGE-L")

All four model configurations were evaluated successfully.
Number of per-case records: 100
Metrics per model: BLEU, ROUGE-1, ROUGE-2, ROUGE-L


In [ ]:


mean_scores = {}

for model_name in MODEL_FIELDS:
    mean_scores[model_name] = {}

    for metric_name in [
        "bleu",
        "rouge1",
        "rouge2",
        "rougeL",
    ]:
        values = [
            record[f"{model_name}_{metric_name}"]
            for record in results
        ]

        mean_scores[model_name][metric_name] = float(
            np.mean(values)
        )




category_mean_scores = {}

for category in sorted(category_counts):
    category_records = [
        record
        for record in results
        if record["category"] == category
    ]

    category_mean_scores[category] = {}

    for model_name in MODEL_FIELDS:
        category_mean_scores[category][model_name] = {}

        for metric_name in [
            "bleu",
            "rouge1",
            "rouge2",
            "rougeL",
        ]:
            values = [
                record[f"{model_name}_{metric_name}"]
                for record in category_records
            ]

            category_mean_scores[category][model_name][
                metric_name
            ] = float(np.mean(values))




version = 1

while True:
    suffix = "" if version == 1 else f"_v{version}"

    PER_CASE_OUTPUT_PATH = (
        RESULT_DIR
        / f"traditional_metrics_unified{suffix}.json"
    )

    SUMMARY_OUTPUT_PATH = (
        RESULT_DIR
        / f"traditional_metrics_unified_summary{suffix}.json"
    )

    if (
        not PER_CASE_OUTPUT_PATH.exists()
        and not SUMMARY_OUTPUT_PATH.exists()
    ):
        break

    version += 1




with open(
    PER_CASE_OUTPUT_PATH,
    "x",
    encoding="utf-8",
) as f:
    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2,
    )



summary_output = {
    "description": (
        "Unified reference-based lexical evaluation of "
        "Base, FT, RAG, and FT+RAG single-turn responses."
    ),
    "input_file": RESPONSES_PATH.name,
    "number_of_cases": len(test_cases),
    "number_of_categories": len(category_counts),
    "cases_per_category": 5,
    "models": list(MODEL_FIELDS.keys()),
    "metric_configuration": {
        "bleu_implementation": (
            "nltk.translate.bleu_score.sentence_bleu"
        ),
        "bleu_tokenisation": (
            "lowercase whitespace-delimited tokens"
        ),
        "bleu_smoothing": (
            "NLTK SmoothingFunction.method4"
        ),
        "rouge_implementation": (
            "rouge_score.rouge_scorer.RougeScorer"
        ),
        "rouge_metrics": [
            "rouge1",
            "rouge2",
            "rougeL",
        ],
        "rouge_score_type": "F-measure",
        "rouge_stemming": True,
    },
    "mean_scores": mean_scores,
    "category_mean_scores": category_mean_scores,
}

with open(
    SUMMARY_OUTPUT_PATH,
    "x",
    encoding="utf-8",
) as f:
    json.dump(
        summary_output,
        f,
        ensure_ascii=False,
        indent=2,
    )




summary_df = pd.DataFrame(mean_scores).T

summary_df.index = [
    "Base",
    "FT",
    "RAG",
    "FT+RAG",
]

summary_df.index.name = "Model"

print("\nUnified single-turn lexical metrics:")
display(summary_df.round(4))

print("\nNew files created:")
print("Per-case metrics:", PER_CASE_OUTPUT_PATH)
print("Summary:", SUMMARY_OUTPUT_PATH)

print("\nOriginal files were not modified.")


Unified single-turn lexical metrics:


,bleu,rouge1,rouge2,rougeL
Model,,,,
Base,0.0141,0.1999,0.0232,0.1266
FT,0.0161,0.1998,0.0268,0.1375
RAG,0.0176,0.2087,0.0332,0.1386
FT+RAG,0.0157,0.2193,0.0268,0.1551



New files created:
Per-case metrics: /content/drive/MyDrive/cbt_results/traditional_metrics_unified.json
Summary: /content/drive/MyDrive/cbt_results/traditional_metrics_unified_summary.json

Original files were not modified.


In [ ]:


OLD_METRICS_PATH = (
    RESULT_DIR / "traditional_metrics.json"
)


NEW_METRICS_PATH = PER_CASE_OUTPUT_PATH

assert OLD_METRICS_PATH.exists()
assert NEW_METRICS_PATH.exists()

with open(
    OLD_METRICS_PATH,
    "r",
    encoding="utf-8",
) as f:
    old_metrics = json.load(f)

with open(
    NEW_METRICS_PATH,
    "r",
    encoding="utf-8",
) as f:
    new_metrics = json.load(f)

old_by_id = {
    record["id"]: record
    for record in old_metrics
}

new_by_id = {
    record["id"]: record
    for record in new_metrics
}

assert set(old_by_id) == set(new_by_id), (
    "The old and new files contain different case IDs."
)

assert len(old_by_id) == 100, (
    f"Expected 100 cases, found {len(old_by_id)}."
)


# RAG was previously missing, so only compare the three
# configurations whose metrics already existed.
MODELS_TO_COMPARE = [
    "base",
    "ft",
    "ft_rag",
]

METRICS_TO_COMPARE = [
    "bleu",
    "rouge1",
    "rouge2",
    "rougeL",
]

comparison_records = []

for case_id in sorted(old_by_id):
    old_record = old_by_id[case_id]
    new_record = new_by_id[case_id]

    for model_name in MODELS_TO_COMPARE:
        for metric_name in METRICS_TO_COMPARE:
            field = f"{model_name}_{metric_name}"

            assert field in old_record, (
                f"{field} is missing from the old file, "
                f"case {case_id}."
            )

            assert field in new_record, (
                f"{field} is missing from the new file, "
                f"case {case_id}."
            )

            old_value = float(old_record[field])
            new_value = float(new_record[field])
            absolute_difference = abs(
                new_value - old_value
            )

            comparison_records.append(
                {
                    "id": case_id,
                    "model": model_name,
                    "metric": metric_name,
                    "old_value": old_value,
                    "new_value": new_value,
                    "absolute_difference": (
                        absolute_difference
                    ),
                }
            )

comparison_df = pd.DataFrame(comparison_records)

TOLERANCE = 1e-12

comparison_summary = (
    comparison_df
    .groupby(["model", "metric"])
    .agg(
        old_mean=("old_value", "mean"),
        new_mean=("new_value", "mean"),
        mean_absolute_difference=(
            "absolute_difference",
            "mean",
        ),
        maximum_absolute_difference=(
            "absolute_difference",
            "max",
        ),
        changed_cases=(
            "absolute_difference",
            lambda values: int(
                (values > TOLERANCE).sum()
            ),
        ),
    )
    .reset_index()
)

print("Comparison of previously existing metrics:")
display(comparison_summary.round(12))

Comparison of previously existing metrics:


,model,metric,old_mean,new_mean,mean_absolute_difference,maximum_absolute_difference,changed_cases
0,base,bleu,0.014066,0.014066,0.0,0.0,0
1,base,rouge1,0.199904,0.199904,0.0,0.0,0
2,base,rouge2,0.023222,0.023222,0.0,0.0,0
3,base,rougeL,0.126594,0.126594,0.0,0.0,0
4,ft,bleu,0.016083,0.016083,0.0,0.0,0
5,ft,rouge1,0.199766,0.199766,0.0,0.0,0
6,ft,rouge2,0.026785,0.026785,0.0,0.0,0
7,ft,rougeL,0.137537,0.137537,0.0,0.0,0
8,ft_rag,bleu,0.015694,0.015694,0.0,0.0,0
9,ft_rag,rouge1,0.219329,0.219329,0.0,0.0,0


In [ ]:
maximum_difference = comparison_df[
    "absolute_difference"
].max()

number_of_changed_values = int(
    (
        comparison_df["absolute_difference"]
        > TOLERANCE
    ).sum()
)

print("Maximum absolute difference:", maximum_difference)
print("Number of changed values:", number_of_changed_values)

if number_of_changed_values == 0:
    print(
        "\nPASS: All previous Base, FT, and FT+RAG "
        "metrics are unchanged."
    )
    print(
        "Only the previously missing RAG metrics "
        "were added."
    )
else:
    print(
        "\nWARNING: Some previous metric values changed."
    )

    changed_df = comparison_df[
        comparison_df["absolute_difference"]
        > TOLERANCE
    ].copy()

    display(
        changed_df
        .sort_values(
            "absolute_difference",
            ascending=False,
        )
        .head(30)
    )

Maximum absolute difference: 0.0
Number of changed values: 0

PASS: All previous Base, FT, and FT+RAG metrics are unchanged.
Only the previously missing RAG metrics were added.


In [ ]:
RAG_FIELDS = [
    "rag_bleu",
    "rag_rouge1",
    "rag_rouge2",
    "rag_rougeL",
]

old_rag_field_counts = {
    field: sum(
        field in record
        for record in old_metrics
    )
    for field in RAG_FIELDS
}

new_rag_field_counts = {
    field: sum(
        field in record
        for record in new_metrics
    )
    for field in RAG_FIELDS
}

rag_field_check = pd.DataFrame(
    {
        "Old file": old_rag_field_counts,
        "New file": new_rag_field_counts,
    }
)

print("RAG metric-field availability:")
display(rag_field_check)

assert all(
    count == 100
    for count in new_rag_field_counts.values()
), "The new file does not contain all RAG metrics."

print("PASS: All four RAG metrics exist for all 100 cases.")

RAG metric-field availability:


,Old file,New file
rag_bleu,0,100
rag_rouge1,0,100
rag_rouge2,0,100
rag_rougeL,0,100


PASS: All four RAG metrics exist for all 100 cases.
